# 05 — Model Evaluation & Explainability
Evaluate the best saved model on the held-out test set and visualise results.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


## 1. Load Dataset

In [ ]:
df = pd.read_csv("data/cleanData.csv")
df.head()


## 2. Define Features & Target

In [ ]:
TARGET = "Machine failure"

FEATURES = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Temp_Diff",
    "Power_Index",
    "Wear_Speed_Ratio",
    "Temperature_Stress",
]

numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Temp_Diff",
    "Power_Index",
    "Wear_Speed_Ratio",
    "Temperature_Stress",
]
categorical_features = ["Type"]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test  shape:", X_test.shape)


## 3. Load Best Model (or retrain if missing / corrupted)

In [ ]:
model_path = Path("models/best_model.pkl")

if not model_path.exists():
    raise FileNotFoundError(
        "models/best_model.pkl not found. "
        "Please run notebook/Model_build.ipynb first."
    )

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    loaded = joblib.load(model_path)

# Guard: old Model_build.ipynb had a bug where it saved the model NAME
# (a string) instead of the pipeline. Detect and retrain automatically.
if isinstance(loaded, str) or not hasattr(loaded, "predict"):
    print(f"WARNING: Loaded object is '{loaded}' (a string). "
          "The model was not saved correctly. Retraining XGBoost now...")

    preprocessor = ColumnTransformer(transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore"))]),
         categorical_features),
    ])

    best_model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=42,
        )),
    ])
    best_model.fit(X_train, y_train)
    joblib.dump(best_model, model_path)
    print("Model retrained and saved correctly.")
else:
    best_model = loaded
    print(f"Model loaded successfully: {type(best_model).__name__}")
    print(f"Steps: {[s[0] for s in best_model.steps]}")


## 4. Predictions on Test Set

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Predictions shape:", y_pred.shape)
print("Sample predictions:", y_pred[:10])


## 5. Evaluation Metrics

In [ ]:
accuracy     = accuracy_score(y_test, y_pred)
precision    = precision_score(y_test, y_pred, zero_division=0)
recall       = recall_score(y_test, y_pred, zero_division=0)
f1           = f1_score(y_test, y_pred, zero_division=0)
roc_auc      = roc_auc_score(y_test, y_prob)
balanced_acc = balanced_accuracy_score(y_test, y_pred)
mcc          = matthews_corrcoef(y_test, y_pred)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score",
                "ROC-AUC", "Balanced Accuracy", "MCC"],
    "Value":  [accuracy, precision, recall, f1,
               roc_auc, balanced_acc, mcc],
})
metrics_df["Value"] = metrics_df["Value"].round(4)
print(metrics_df.to_string(index=False))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Healthy", "Failure"]))


## 6. Confusion Matrix & Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
cm_disp = ConfusionMatrixDisplay(cm, display_labels=["Healthy", "Failure"])
cm_disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix")

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title("ROC Curve")
axes[1].plot([0, 1], [0, 1], "k--", label="Random")
axes[1].legend()

# Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[2])
axes[2].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.show()


## 7. Feature Importances

In [ ]:
final_estimator = best_model.steps[-1][1]
importances = getattr(final_estimator, "feature_importances_", None)

if importances is not None:
    # The ColumnTransformer puts numeric cols first, then OHE cols for Type
    preprocessor_step = best_model.named_steps["preprocessor"]
    try:
        ohe_cols = preprocessor_step.named_transformers_["cat"]["encoder"] \
                       .get_feature_names_out(["Type"]).tolist()
    except Exception:
        ohe_cols = ["Type_H", "Type_L", "Type_M"]

    feat_names = numeric_features + ohe_cols
    imp_df = pd.DataFrame({"Feature": feat_names, "Importance": importances})

    # Aggregate OHE Type columns back to "Type"
    imp_dict = {}
    for name, imp in zip(feat_names, importances):
        base = "Type" if name.startswith("Type") else name
        imp_dict[base] = imp_dict.get(base, 0.0) + float(imp)

    agg_df = pd.DataFrame(imp_dict.items(), columns=["Feature", "Importance"])
    agg_df = agg_df.sort_values("Importance", ascending=True)

    plt.figure(figsize=(9, 5))
    plt.barh(agg_df["Feature"], agg_df["Importance"], color="steelblue")
    plt.xlabel("Importance")
    plt.title("Feature Importance — Best Model")
    plt.tight_layout()
    plt.show()
    print(agg_df.sort_values("Importance", ascending=False).to_string(index=False))
else:
    print("This model type does not expose feature_importances_.")


## 8. Model Summary

In [ ]:
print("=== Model Summary ===")
print(f"Type   : {type(best_model).__name__}")
print(f"Steps  : {[s[0] for s in best_model.steps]}")
print()
print("Pipeline:")
print(best_model)
